# 02 — EDA and Cleaning

**IBM Bob assisted** — All EDA code generated via IBM Bob Phase 2.

Null analysis · Distribution plots · Correlation heatmap · Time-series overlays

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

RAW_DIR = Path('../data/raw')
PROC_DIR = Path('../data/processed')
PROC_DIR.mkdir(exist_ok=True)

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Libraries loaded')

## Load Raw Data

In [ ]:
# Find most recent donki files
def load_donki(event: str) -> pd.DataFrame:
    files = sorted(RAW_DIR.glob(f'donki_{event.lower()}_*.json'), reverse=True)
    if not files:
        print(f'  [warn] No {event} files found, returning empty DataFrame')
        return pd.DataFrame()
    with open(files[0]) as f:
        data = json.load(f)
    return pd.DataFrame(data) if data else pd.DataFrame()

flr_df  = load_donki('FLR')
cme_df  = load_donki('CME')
gst_df  = load_donki('GST')
sep_df  = load_donki('SEP')
launches = pd.read_csv(RAW_DIR / 'launch_history.csv', parse_dates=['launch_date'])

# Kp index
kp_raw = json.loads((RAW_DIR / 'swpc_kp_1m.json').read_text())
kp_df  = pd.DataFrame(kp_raw)

# Solar flux
flux_raw = json.loads((RAW_DIR / 'swpc_solar_cycle_indices.json').read_text())
flux_df  = pd.DataFrame(flux_raw)

print('FLR:', len(flr_df), '  CME:', len(cme_df), '  GST:', len(gst_df), '  SEP:', len(sep_df))
print('Kp rows:', len(kp_df), '  Flux rows:', len(flux_df))

## Parse and Clean Timestamps

In [ ]:
def parse_ts(df: pd.DataFrame, col: str) -> pd.DataFrame:
    df = df.copy()
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')
    return df

if not flr_df.empty and 'beginTime' in flr_df.columns:
    flr_df = parse_ts(flr_df, 'beginTime')
    flr_df['date'] = flr_df['beginTime'].dt.normalize()

if not cme_df.empty and 'startTime' in cme_df.columns:
    cme_df = parse_ts(cme_df, 'startTime')
    cme_df['date'] = cme_df['startTime'].dt.normalize()

if not gst_df.empty and 'startTime' in gst_df.columns:
    gst_df = parse_ts(gst_df, 'startTime')
    gst_df['date'] = gst_df['startTime'].dt.normalize()

# Parse Kp — SWPC format: time_tag field
if 'time_tag' in kp_df.columns:
    kp_df['datetime'] = pd.to_datetime(kp_df['time_tag'], utc=True, errors='coerce')
    kp_df['date'] = kp_df['datetime'].dt.normalize()
    kp_val_col = [c for c in kp_df.columns if 'kp' in c.lower()]
    if kp_val_col:
        kp_df[kp_val_col[0]] = pd.to_numeric(kp_df[kp_val_col[0]], errors='coerce')
        KP_COL = kp_val_col[0]
        print('Kp column:', KP_COL)
    else:
        KP_COL = None
else:
    KP_COL = None

# Build daily Kp mean
if KP_COL:
    kp_daily = kp_df.groupby('date')[KP_COL].mean().reset_index()
    kp_daily.rename(columns={KP_COL: 'kp_mean'}, inplace=True)
else:
    kp_daily = pd.DataFrame(columns=['date','kp_mean'])

# Parse F10.7 — look for 'f10.7' or 'ssn' column
flux_col = [c for c in flux_df.columns if '10.7' in c.lower() or 'flux' in c.lower()]
date_col  = [c for c in flux_df.columns if 'time' in c.lower() or 'date' in c.lower() or 'year' in c.lower()]
print('Flux cols:', flux_df.columns.tolist())
if flux_col and date_col:
    flux_df['date'] = pd.to_datetime(flux_df[date_col[0]], errors='coerce', utc=True).dt.normalize()
    flux_df['f10_7'] = pd.to_numeric(flux_df[flux_col[0]], errors='coerce')
    flux_daily = flux_df[['date','f10_7']].dropna()
else:
    flux_daily = pd.DataFrame(columns=['date','f10_7'])
print('Daily Kp rows:', len(kp_daily), '  Daily flux rows:', len(flux_daily))

## Null Analysis

In [ ]:
for name, df in [('FLR', flr_df),('CME', cme_df),('GST', gst_df),('Kp daily', kp_daily),('Flux', flux_daily)]:
    if df.empty:
        print(f'{name}: EMPTY')
        continue
    null_pct = (df.isnull().sum() / len(df) * 100).round(1)
    print(f'\n--- {name} ---')
    print(null_pct[null_pct > 0].to_string() or '  (no nulls)')

## Kp Index Distribution

In [ ]:
if not kp_daily.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(kp_daily['kp_mean'].dropna(), bins=30, color='#0f62fe', edgecolor='white')
    axes[0].set_title('Kp Index Distribution (daily mean)', fontsize=13)
    axes[0].set_xlabel('Kp Index'); axes[0].set_ylabel('Count')
    
    kp_daily.set_index('date')['kp_mean'].plot(ax=axes[1], color='#0f62fe', alpha=0.8, lw=0.8)
    axes[1].set_title('Daily Mean Kp Index over Time', fontsize=13)
    axes[1].set_ylabel('Kp')
    plt.tight_layout()
    plt.savefig('../data/processed/eda_kp_distribution.png', bbox_inches='tight')
    plt.show()
else:
    print('No Kp data available for plotting')

## Solar Flare Class Distribution

In [ ]:
if not flr_df.empty:
    class_col = [c for c in flr_df.columns if 'class' in c.lower()]
    if class_col:
        flr_class = flr_df[class_col[0]].str[0].value_counts().sort_index()
        fig = px.bar(x=flr_class.index, y=flr_class.values,
                     labels={'x':'Flare Class','y':'Count'},
                     title='Solar Flare Count by Class', color_discrete_sequence=['#0f62fe'])
        fig.show()
    else:
        print('No class column in FLR data')
else:
    print('No FLR data')

## CME Speed Distribution

In [ ]:
if not cme_df.empty:
    # Extract speed from nested cmeAnalyses if present
    speeds = []
    for row in cme_df.to_dict('records'):
        analyses = row.get('cmeAnalyses') or []
        if analyses and isinstance(analyses, list):
            for a in analyses:
                if isinstance(a, dict) and a.get('speed'):
                    speeds.append(float(a['speed']))
    if speeds:
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.hist(speeds, bins=40, color='#ee538b', edgecolor='white')
        ax.set_title('CME Speed Distribution (km/s)', fontsize=13)
        ax.set_xlabel('Speed (km/s)'); ax.set_ylabel('Count')
        plt.tight_layout()
        plt.savefig('../data/processed/eda_cme_speed.png', bbox_inches='tight')
        plt.show()
    else:
        print('CME speed data not available in nested structure')
else:
    print('No CME data')

## Correlation Heatmap

In [ ]:
# Build a merged daily dataframe for correlation analysis
import functools

dfs_to_merge = []
if not kp_daily.empty:
    dfs_to_merge.append(kp_daily.copy())

if not flux_daily.empty:
    merged_df = flux_daily.copy() if not dfs_to_merge else functools.reduce(
        lambda a, b: pd.merge(a, b, on='date', how='outer'), dfs_to_merge + [flux_daily.copy()])
    dfs_to_merge = [merged_df]

if dfs_to_merge:
    combined = dfs_to_merge[0].select_dtypes(include='number')
    if combined.shape[1] > 1:
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(combined.corr(), annot=True, fmt='.2f', cmap='coolwarm',
                    linewidths=0.5, ax=ax)
        ax.set_title('Feature Correlation Heatmap', fontsize=13)
        plt.tight_layout()
        plt.savefig('../data/processed/eda_correlation_heatmap.png', bbox_inches='tight')
        plt.show()
    else:
        print('Not enough numeric columns for correlation heatmap')
else:
    print('No data available for correlation heatmap')

## Kp Time Series with Launch Events Overlay

In [ ]:
if not kp_daily.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=kp_daily['date'], y=kp_daily['kp_mean'],
        name='Kp Index (daily mean)', line=dict(color='#0f62fe', width=1.5)))

    # Storm threshold lines
    for level, color, label in [(5,'orange','G1 Storm Threshold'),
                                  (7,'red','G3 Storm Threshold')]:
        fig.add_hline(y=level, line_dash='dash', line_color=color,
                      annotation_text=label, annotation_position='top right')

    # Launch event markers
    go_launches   = launches[launches['launch_go']==1]
    scrub_launches = launches[launches['launch_go']==0]

    fig.add_trace(go.Scatter(
        x=go_launches['launch_date'], y=[0.2]*len(go_launches),
        mode='markers', marker=dict(symbol='triangle-up', size=10, color='green'),
        name='Launch GO'))

    fig.add_trace(go.Scatter(
        x=scrub_launches['launch_date'], y=[0.1]*len(scrub_launches),
        mode='markers', marker=dict(symbol='x', size=8, color='crimson'),
        name='Launch SCRUB'))

    fig.update_layout(title='Kp Index with Launch Events',
                      xaxis_title='Date', yaxis_title='Kp Index',
                      legend=dict(orientation='h', y=1.02),
                      height=400)
    fig.show()
else:
    print('No Kp data for time series')

## Save Cleaned Master DataFrame

In [ ]:
# We store intermediate daily master for feature engineering
if not kp_daily.empty:
    master = kp_daily.copy()
    if not flux_daily.empty:
        master = master.merge(flux_daily, on='date', how='left')
    master = master.merge(launches.rename(columns={'launch_date':'date'}), on='date', how='left')
    master.to_parquet(PROC_DIR / 'daily_master.parquet', index=False)
    print(f'Saved daily_master.parquet: {master.shape}')
    display(master.head())